### データの読み込み

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()       # 手書き文字画像データの読み込み
print(dir(digits))           # データセットの内容

print(digits.data.shape)     # 画像のピクセル値配列の大きさ
print(digits.target.shape)   # 画像の数字ラベル配列の大きさ

print(digits.data[0])        # 1枚目の画像ピクセル値
print(digits.target[0])      # 1枚目の画像ラベル

In [ ]:
import matplotlib.pyplot as plt 

# 1枚目の画像ファイル表示
plt.imshow(digits.data[0].reshape(8, 8), cmap='gray')
plt.show()

### 特徴量の作成

In [ ]:
import pandas as pd

# ピクセル値配列をデータフレーム型へ変換
digits = load_digits()
input_df = pd.DataFrame(digits.data)

print(input_df.shape)   # ピクセル値配列の大きさ
input_df.head()         # データの先頭5行を表示

In [ ]:
import numpy as np
import random

max_px_vals = digits.data.max()    # ピクセルの最大値
threshold = max_px_vals/2          # 最大値の半分を閾値とする

# 各画像について、ピクセル値をランダムに欠損させる
for i in range(input_df.shape[0]):
    # 閾値を超える列idを取り出し
    sel_col_ids = np.where(input_df.iloc[i, :]>threshold)[0]
    # ランダムに3つの列idを取り出し
    sel_col_ids = random.sample(sel_col_ids.tolist(), k=3)
    
    for s in sel_col_ids:
        input_df.iloc[i, s] = 0    # 選択した列に値0を代入

In [ ]:
# 欠損させた画像を1枚選択して可視化
plt.imshow(np.array(input_df.iloc[0, :]).reshape(8, 8), cmap='gray')
plt.show()

### 目的変数の作成

In [ ]:
# ピクセル値配列をデータフレーム型へ変換
digits = load_digits()
output_df = pd.DataFrame(digits.data)

print(output_df.shape)   # ピクセル値配列の大きさ
output_df.head()         # データの先頭5行を表示

In [ ]:
# 画像を1枚選択して可視化
plt.imshow(np.array(output_df.iloc[0, :]).reshape(8, 8), cmap='gray')
plt.show()

### 学習と評価データを作成

In [ ]:
# 全体の8割を学習データとして分割
trainX = input_df.iloc[:1438, :]/max_px_vals
trainY = output_df.iloc[:1438, :]/max_px_vals

print(trainX.shape, trainY.shape)

In [ ]:
# 全体の2割を評価データとして分割
testX = input_df.iloc[1438:, :]/max_px_vals
testY = output_df.iloc[1438:, :]/max_px_vals

print(testX.shape, testY.shape)

### 回帰モデルの学習

In [ ]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# オートエンコーダを作成
model = Sequential()
# エンコード
model.add(Dense(32, activation='relu', input_shape=(64,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
# デコード
model.add(Dense(16, activation='relu'))
model.add(Dense(32, activation='relu'))
# 出力層
model.add(Dense(64, activation='sigmoid'))

# 作成したネットワークの確認
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# 学習条件の設定
model.compile(loss='mse', optimizer='adam')
early_stopping = EarlyStopping(monitor='val_loss', patience=50)

# 学習の実行
hist = model.fit(trainX, trainY, batch_size=64, verbose=1, 
                 epochs=1000, validation_split=0.2, callbacks=[early_stopping])

In [ ]:
# 誤差の収束過程を描画
plt.plot(hist.history['loss'], label='loss')           # 訓練データ
plt.plot(hist.history['val_loss'], label='val_loss')   # テストデータ

plt.xlabel('epoch')   # 横軸ラベルを追加
plt.ylabel('loss')    # 縦軸ラベルを追加
plt.legend()          # 凡例を追加
plt.show()

### 回帰モデルの評価

In [ ]:
# モデルを評価データに適用
pred = model.predict(testX)
pred = np.array(pred*max_px_vals, dtype='int')

# 出力画像を1枚選択して可視化
plt.imshow(pred[0].reshape(8, 8), cmap='gray')
plt.show()